<a href="https://colab.research.google.com/github/darlingloya13-cmd/Zudio-prediction-ML-model-/blob/main/new_zudio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression,Lasso, Ridge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df_main_model=pd.read_csv('new zudio1.csv')
df_main_model.head()

,State,City,Existing_Stores,Estimated_Population,Per_Capita_Income,Malls,buyers_count,State_Encoded,City_Encoded
0,Andhra Pradesh,Adoni,1,261000,219518,1,130500.0,0,0
1,Andhra Pradesh,Anantapur,2,498089,219518,1,249044.5,0,16
2,Andhra Pradesh,Chirala,1,229000,219518,1,114500.0,0,63
3,Andhra Pradesh,Chittoor,1,248000,219518,1,124000.0,0,65
4,Andhra Pradesh,Eluru,1,355000,219518,1,177500.0,0,85


In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Initialize the MinMaxScaler
scaler = MinMaxScaler()

# Columns to normalize
columns_to_normalize = ['Estimated_Population', 'buyers_count', 'Existing_Stores']

# Apply Min-Max Scaling
df_main_model[columns_to_normalize] = scaler.fit_transform(df_main_model[columns_to_normalize])

# Rename the normalized columns (or create new ones with 'Norm_' prefix)
df_main_model['Norm_Population'] = df_main_model['Estimated_Population']
df_main_model['Norm_Buyers'] = df_main_model['buyers_count']
df_main_model['Norm_Stores'] = df_main_model['Existing_Stores']

print("Normalized columns added successfully.")
print(df_main_model[["Norm_Population", "Norm_Buyers", "Norm_Stores"]].head())

Normalized columns added successfully.
   Norm_Population  Norm_Buyers  Norm_Stores
0         0.015339     0.015339     0.000000
1         0.029465     0.029465     0.015625
2         0.013433     0.013433     0.000000
3         0.014565     0.014565     0.000000
4         0.020940     0.020940     0.000000


In [ ]:
df.shape

(307, 9)

In [ ]:

df_main_model['Priority_Index'] = (
    (df_main_model['Norm_Population'] * 0.4) +
    (df_main_model['Norm_Buyers'] * 0.4) -
    (df_main_model['Norm_Stores'] * 0.2) + 0.5  # Adding 0.5 shifts everything above zero!
)

print("New Min:", df_main_model['Priority_Index'].min())
print("New Max:", df_main_model['Priority_Index'].max())
print("Zero count:", (df_main_model['Priority_Index'] == 0).sum())

New Min: 0.4222245098543302
New Max: 1.26875
Zero count: 0


In [ ]:
df_main_model.head()

,State,City,Existing_Stores,Estimated_Population,Per_Capita_Income,Malls,buyers_count,State_Encoded,City_Encoded,Norm_Population,Norm_Buyers,Norm_Stores,Priority_Index
0,Andhra Pradesh,Adoni,0.000000,0.015339,219518,1,0.015339,0,0,0.015339,0.015339,0.000000,0.512271
1,Andhra Pradesh,Anantapur,0.015625,0.029465,219518,1,0.029465,0,16,0.029465,0.029465,0.015625,0.520447
2,Andhra Pradesh,Chirala,0.000000,0.013433,219518,1,0.013433,0,63,0.013433,0.013433,0.000000,0.510746
3,Andhra Pradesh,Chittoor,0.000000,0.014565,219518,1,0.014565,0,65,0.014565,0.014565,0.000000,0.511652
4,Andhra Pradesh,Eluru,0.000000,0.020940,219518,1,0.020940,0,85,0.020940,0.020940,0.000000,0.516752


In [ ]:
df_main_model[['City', 'State', 'Priority_Index']].sort_values(by='Priority_Index', ascending=False)
with pd.option_context('display.max_rows', None):
    print(df_main_model[['City', 'State', 'Priority_Index']])

                           City              State  Priority_Index
0                         Adoni     Andhra Pradesh        0.512271
1                     Anantapur     Andhra Pradesh        0.520447
2                       Chirala     Andhra Pradesh        0.510746
3                      Chittoor     Andhra Pradesh        0.511652
4                         Eluru     Andhra Pradesh        0.516752
5                      Guntakal     Andhra Pradesh        0.508315
6                        Guntur     Andhra Pradesh        0.541519
7                    Hindupuram     Andhra Pradesh        0.510031
8                        Kadapa     Andhra Pradesh        0.510031
9                      Kakinada     Andhra Pradesh        0.523016
10                      Kurnool     Andhra Pradesh        0.535385
11                      Nellore     Andhra Pradesh        0.539757
12                       Ongole     Andhra Pradesh        0.510719
13                Payakaraopeta     Andhra Pradesh        0.50

In [ ]:
# Filter cities with Priority_Index > 0.6 and sort from highest to lowest
high_priority_df = df_main_model[df_main_model['Priority_Index'] > 0.6][['City', 'State', 'Priority_Index']].sort_values(by='Priority_Index', ascending=False)

# Display interactive table in Colab
from google.colab import data_table
data_table.enable_dataframe_formatter()

high_priority_df

,City,State,Priority_Index
51,Delhi,Delhi,1.268750
294,Barrackpore,West Bengal,0.867676
293,Bardhaman,West Bengal,0.867676
282,Prayagraj,Uttar Pradesh,0.783637
229,Jaipur,Rajasthan,0.759407
...,...,...,...
284,Sambhal,Uttar Pradesh,0.605480
142,Bhopal,Madhya Pradesh,0.603469
204,Amritsar,Punjab,0.602919
122,Alappuzha,Kerala,0.601249


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import pandas as pd

# 1. Define your Features (X) and Target (y)
# Ensure your columns match what is currently inside df_main_model
features = ['Estimated_Population', 'Per_Capita_Income', 'Existing_Stores', 'buyers_count', 'State_Encoded', 'City_Encoded']
X = df_main_model[features]
y = df_main_model['Priority_Index'] # or 'Priority_Index' depending on your column naming

# 2. Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize and train the Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Predict on the test set and evaluate performance
y_pred = model.predict(X_test)

print("--- Model Performance ---")
print("R2 Score:", r2_score(y_test, y_pred))
print("Mean Squared Error (MSE):", mean_squared_error(y_test, y_pred))


--- Model Performance ---
R2 Score: 0.9827524925412927
Mean Squared Error (MSE): 5.151695599000358e-05


In [ ]:
from sklearn.linear_model import LinearRegression

# Initialize and train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict and evaluate
y_pred_lr = lr_model.predict(X_test)
print("Linear Regression R2 Score:", r2_score(y_test, y_pred_lr))

NameError: name 'X_train' is not defined

In [ ]:
df=pd.read_csv('no zudio store.csv') # Changed from .xlsx to .csv
df.head()

,city,State,population,buyers count,Malls
0,Takkolu,Andhra Pradesh,3465,1732.5,0
1,Satanagaram,Andhra Pradesh,5447,2723.5,0
2,Patacudapah,Andhra Pradesh,4730,2365.0,0
3,Guntar,Andhra Pradesh,670073,335036.5,1
4,Patapadu,Andhra Pradesh,3933,1966.5,0


In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

In [ ]:
df_new_cities=pd.read_csv('no zudio store.csv', index_col=False) # Ensure 'city' column is not dropped as index
df_new_cities.head()

,city,State,population,buyers count,Malls
0,Takkolu,Andhra Pradesh,3465,1732.5,0
1,Satanagaram,Andhra Pradesh,5447,2723.5,0
2,Patacudapah,Andhra Pradesh,4730,2365.0,0
3,Guntar,Andhra Pradesh,670073,335036.5,1
4,Patapadu,Andhra Pradesh,3933,1966.5,0


In [ ]:

print(df_new_cities.head())

           city           State  population  buyers count  Malls
0       Takkolu  Andhra Pradesh        3465        1732.5      0
1  Satanagaram  Andhra Pradesh        5447        2723.5      0
2  Patacudapah  Andhra Pradesh        4730        2365.0      0
3        Guntar  Andhra Pradesh      670073      335036.5      1
4    Patapadu  Andhra Pradesh        3933        1966.5      0


In [ ]:
# 1. First, ensure the 'Existing_Stores' column exists in your raw dataframe
df_new_cities['Existing_Stores'] = 0

# 2. Corrected groupby and aggregation block
df_new_cities_grouped = df_new_cities.groupby(['city', 'State']).agg(
    Estimated_Population=('population', 'first'),
    Existing_Stores=('Existing_Stores', 'first'),
    buyers_count=('buyers count', 'first'),
    Malls=('Malls', 'first')
).reset_index()

df_new_cities_grouped.head()

,city,State,Estimated_Population,Existing_Stores,buyers_count,Malls
0,Abhepur,Punjab,1895686,0,947843.0,0
1,Agaram,Tamil Nadu,3949,0,1974.5,0
2,Alleppey,Kerala,240991,0,120495.5,1
3,Alwal,Telangana,240000,0,120000.0,1
4,Amarnath,Maharashtra,253475,0,126737.5,0


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Initialize encoders
state_encoder = LabelEncoder()
city_encoder = LabelEncoder()

# Transform State and City columns into numerical values
df_new_cities_grouped['State_Encoded'] = state_encoder.fit_transform(df_new_cities_grouped['State'])
df_new_cities_grouped['City_Encoded'] = city_encoder.fit_transform(df_new_cities_grouped['city'])

# View the result
df_new_cities_grouped.head()

,city,State,Estimated_Population,Existing_Stores,buyers_count,Malls,State_Encoded,City_Encoded
0,Abhepur,Punjab,1895686,0,947843.0,0,17,0
1,Agaram,Tamil Nadu,3949,0,1974.5,0,18,1
2,Alleppey,Kerala,240991,0,120495.5,1,12,2
3,Alwal,Telangana,240000,0,120000.0,1,19,3
4,Amarnath,Maharashtra,253475,0,126737.5,0,14,4


In [ ]:
# Add the Per_Capita_Income column filled with 0s
df_new_cities_grouped['Per_Capita_Income'] = 0

# Now your feature list will match what the model expects:
features = ['Estimated_Population', 'Per_Capita_Income', 'Existing_Stores', 'buyers_count', 'State_Encoded', 'City_Encoded']

# Run your prediction
df_new_cities_grouped['Predicted_Priority'] = lr_model.predict(df_new_cities_grouped[features])

NotFittedError: This LinearRegression instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [ ]:
df_new_cities_grouped.head()

,city,State,Estimated_Population,Existing_Stores,buyers_count,Malls,State_Encoded,City_Encoded,Per_Capita_Income
0,Abhepur,Punjab,1895686,0,947843.0,0,17,0,0
1,Agaram,Tamil Nadu,3949,0,1974.5,0,18,1,0
2,Alleppey,Kerala,240991,0,120495.5,1,12,2,0
3,Alwal,Telangana,240000,0,120000.0,1,19,3,0
4,Amarnath,Maharashtra,253475,0,126737.5,0,14,4,0


In [ ]:
# 1. Add the missing Per_Capita_Income column filled with 0s (since your model expects it)
df_new_cities_grouped['Per_Capita_Income'] = 0

# 2. Extract the exact feature set your model was trained on
features = ['Estimated_Population', 'Per_Capita_Income', 'Existing_Stores', 'buyers_count', 'State_Encoded', 'City_Encoded']
X_new = df_new_cities_grouped[features]

# 3. Predict the Priority scores using your trained Linear Regression model
df_new_cities_grouped['Predicted_Priority'] = lr_model.predict(X_new)

# 4. Sort the cities from highest priority to lowest to see the best expansion spots
df_new_cities_ranked = df_new_cities_grouped.sort_values(by='Predicted_Priority', ascending=False)

# 5. Display the top 10 best cities for a new Zudio store
print(df_new_cities_ranked[['city', 'State', 'Predicted_Priority']].head(10))

# 6. Save the results to a new CSV file
df_new_cities_ranked.to_csv('zudio_expansion_ranked_cities.csv', index=False)

NotFittedError: This LinearRegression instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [ ]:
# Filter cities where Predicted_Priority > 0.6
high_priority_cities = df_new_cities_ranked[df_new_cities_ranked['Predicted_Priority'] > 0.6]

# Display the filtered list with city, State, and Predicted_Priority
print(high_priority_cities[['city', 'State', 'Predicted_Priority']])

# Optional: Print how many cities meet this criteria
print(f"\nTotal high-priority cities found: {len(high_priority_cities)}")

NameError: name 'df_new_cities_ranked' is not defined

In [ ]:
import joblib

joblib.dump(model, 'zudio_rf_model.pkl')

['zudio_rf_model.pkl']

In [ ]:
import joblib

joblib.dump(model, 'zudio_rf_model.pkl') # Corrected from 'rf' to 'model'

['zudio_rf_model.pkl']